In [19]:
import sys
sys.path.append("../IMAGE-Mat/scripts/vema")
from preprocessing import preprocessing
from typing import Callable
import prism
from pathlib import Path

In [20]:
base_dir = "../IMAGE-Mat/scripts/vema"
prep_fp = Path("prep_vema.nc")

In [23]:
from survival import convert_life_time_vehicles
import xarray as xr

def export_to_netcdf(prep_data, out_fp):
    simple_datasets = {key: val for key, val in prep_data.items() if key.endswith("_simple")}
    typical_datasets = {key: val for key, val in prep_data.items() if key.endswith("_typical")}
    lf_vehicles = convert_life_time_vehicles(prep_data["lifetimes_vehicles"])
    other_datasets = {key: val for key, val in prep_data.items()
                      if key not in ["lifetimes_vehicles"] + list(simple_datasets) + list(typical_datasets)}
    xr.Dataset(simple_datasets).to_netcdf(out_fp, group="simple", engine="netcdf4")
    xr.Dataset(typical_datasets).to_netcdf(out_fp, group="typical", mode="a", engine="netcdf4")
    xr.Dataset(lf_vehicles).to_netcdf(out_fp, group="lifetimes", mode="a", engine="netcdf4")
    xr.Dataset(other_datasets).to_netcdf(out_fp, group="other", mode="a", engine="netcdf4")
    
def import_from_netcdf(in_fp):
    return {
        "simple": xr.open_dataset(in_fp, group="simple", engine="netcdf4").load(),
        "typical": xr.open_dataset(in_fp, group="typical", engine="netcdf4").load(),
        "lifetimes": xr.open_dataset(in_fp, group="lifetimes", engine="netcdf4").load(),
        "other": xr.open_dataset(in_fp, group="other", engine="netcdf4").load(),
    }

In [22]:
if not prep_fp.is_file():
    _, prep_data = preprocessing(base_dir)
    export_to_netcdf(prep_data, prep_fp)
prep_data = import_from_netcdf("test.nc")


In [ ]:
dist_data_arrays = {}
for dist_name, dist_data in convert_life_time_vehicles(prep_data["lifetimes_vehicles"]).items():
    dist_dict = dist_data[0]
    print(dist_name, list(dist_dict))
    is_array = [key for key, value in dist_dict.items() if isinstance(value, xr.DataArray)]
    cur_data_array = xr.DataArray(0.0, dims=is_array, coords=dist_dict[is_array[0]].coords)
    print(dist_dict["c"].coords)
    

weibull ['c', 'scale', 'loc']
Coordinates:
  * time     (time) int64 2kB 1807 1808 1809 1810 1811 ... 2057 2058 2059 2060
  * mode     (mode) <U4 16B 'Cars'
folded_norm ['c', 'scale', 'loc']
Coordinates:
  * time     (time) int64 2kB 1807 1808 1809 1810 1811 ... 2057 2058 2059 2060
  * mode     (mode) <U25 2kB 'Passenger Planes' 'Freight Planes' ... 'Bikes'


In [ ]:
import xarray as xr

df_names = ["total_nr_vehicles_simple", "material_fractions_simple", "material_fractions_typical", "vehicle_weights_simple",
            "vehicle_weights_typical", "battery_weights_typical"]
sub_prep_data = {key: prep_data[key] for key in df_names}
xr.Dataset(sub_prep_data)

<xarray.Dataset> Size: 6MB
Dimensions:                     (year: 254, mode: 53, region: 28, cohort: 254,
                                 material: 14)
Coordinates:
  * year                        (year) int64 2kB 1807 1808 1809 ... 2059 2060
  * mode                        (mode) <U35 7kB 'Bikes' ... 'Very Large Ships'
  * region                      (region) <U2 224B '1' '10' '11' ... '7' '8' '9'
  * cohort                      (cohort) int64 2kB 1807 1808 1809 ... 2059 2060
  * material                    (material) <U9 504B 'Aluminium' 'Co' ... 'Wood'
Data variables:
    total_nr_vehicles_simple    (year, mode, region) float64 3MB 0.0 ... 122.6
    material_fractions_simple   (cohort, mode, material) float64 2MB 0.46 ......
    material_fractions_typical  (cohort, mode, material) float64 2MB nan ... nan
    vehicle_weights_simple      (cohort, mode) float64 108kB 17.2 nan ... nan
    vehicle_weights_typical     (cohort, mode) float64 108kB nan nan ... nan nan
    battery_weights_typical     (cohort, mode) float64 108kB nan nan ... nan nan

In [ ]:
vehicle_nr = prep_data['total_nr_vehicles_simple']
life_time_vehicles = prep_data["lifetimes_vehicles"]

In [ ]:
from survival import ScipySurvival, SurvivalMatrix
from stock import compute_historic, compute_dynamic_stock_driven
import numpy as np
       
survival_matrix = SurvivalMatrix(ScipySurvival.from_lifetime_vehicles(life_time_vehicles))

In [ ]:
start_simulation = 1970
end_simulation = vehicle_nr.coords["time"].max()
Region = prism.NewDim("region", coords=[str(x) for x in np.arange(1, 29)])
Mode = prism.NewDim("mode", coords=[str(x) for x in vehicle_nr["mode"].to_numpy()])
Cohort = prism.NewDim("cohort", coords=[str(x) for x in vehicle_nr["time"].to_numpy()])

In [ ]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix
    stock: prism.TimeVariable[Region, Mode] #TODO check how to have property that can be both input and output within prism
    stock_function: Callable    # defines the stock function to use e.g. stock or inflow driven

    stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, "count"]         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, "count"] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.stock, self.survival_matrix, self.start_simulation,
                         self.stock_by_cohort, self.inflow, self.outflow_by_cohort,
                         self.stock_function)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        self.stock_function(self.stock, self.stock_by_cohort,  self.inflow, self.outflow_by_cohort, self.survival_matrix, t)

In [ ]:
timeline = prism.Timeline(start=vehicle_nr.coords["time"][0],
                          end=end_simulation, stepsize=1)
timeline_simulate = prism.Timeline(start=start_simulation,
                          end=end_simulation, stepsize=1)

In [ ]:
model = Stocks(timeline, start_simulation=start_simulation, survival_matrix=survival_matrix, stock=vehicle_nr, stock_function = compute_dynamic_stock_driven)

In [ ]:
model.simulate(timeline_simulate)